In [2]:
# =============================================================================
# 06_experiment_tracking.ipynb – MLflow Experiment Tracking
# Projekt: MSIT Mock Interview – Loan Status Klassifikation
# Inhalt:
#   - Alle Modelle mit Parametern & Metriken loggen
#   - Champion v1 als Production registrieren
#   - Artefakte speichern (Plots, CSVs, Modelle)
#   - Experiment-Vergleich via MLflow UI
# =============================================================================

# --- Standard ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# --- MLflow ---
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
from mlflow.models.signature import infer_signature

# --- Scikit-learn ---
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    matthews_corrcoef
)

# --- Joblib ---
import joblib

# --- Ausgabepfade ---
Path("../reports/figures").mkdir(parents=True, exist_ok=True)
Path("../reports/modeling_csv").mkdir(parents=True, exist_ok=True)
Path("../models").mkdir(parents=True, exist_ok=True)

# --- MLflow Tracking URI ---
mlflow.set_tracking_uri("../mlruns")

# --- Experiment definieren ---
EXPERIMENT_NAME = "MSIT_MockInterview_LoanStatus"
mlflow.set_experiment(EXPERIMENT_NAME)

print("✅ MLflow Setup abgeschlossen.")
print(f"   Tracking URI: ../mlruns")
print(f"   Experiment:   {EXPERIMENT_NAME}")
print(f"   MLflow Version: {mlflow.__version__}")

2026/04/21 21:32:45 INFO mlflow.tracking.fluent: Experiment with name 'MSIT_MockInterview_LoanStatus' does not exist. Creating a new experiment.


✅ MLflow Setup abgeschlossen.
   Tracking URI: ../mlruns
   Experiment:   MSIT_MockInterview_LoanStatus
   MLflow Version: 3.11.1


In [3]:
# =============================================================================
# Zelle 2 – Daten & Modelle laden
# =============================================================================

# --- Daten laden ---
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

# --- Champion v1 laden ---
champion_v1 = joblib.load("../models/champion_lgbm_full_v1_final.joblib")

# --- Champion v2 laden ---
champion_v2 = joblib.load("../models/lgbm_v2_gap_penalty.joblib")

print(f"✅ Daten geladen:")
print(f"   X_train: {X_train.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"\n✅ Modelle geladen:")
print(f"   Champion v1: {type(champion_v1).__name__}")
print(f"   Champion v2: {type(champion_v2).__name__}")

# --- Predictions ---
y_pred_v1       = champion_v1.predict(X_test)
y_pred_proba_v1 = champion_v1.predict_proba(X_test)[:, 1]

y_pred_v2       = champion_v2.predict(X_test)
y_pred_proba_v2 = champion_v2.predict_proba(X_test)[:, 1]

# --- Hilfsfunktion: Metriken berechnen ---
def compute_metrics(y_true, y_pred, y_proba):
    return {
        "precision":  round(precision_score(y_true, y_pred), 4),
        "recall":     round(recall_score(y_true, y_pred), 4),
        "f1":         round(f1_score(y_true, y_pred), 4),
        "roc_auc":    round(roc_auc_score(y_true, y_proba), 4),
        "pr_auc":     round(average_precision_score(y_true, y_proba), 4),
        "mcc":        round(matthews_corrcoef(y_true, y_pred), 4),
    }

metrics_v1 = compute_metrics(y_test, y_pred_v1, y_pred_proba_v1)
metrics_v2 = compute_metrics(y_test, y_pred_v2, y_pred_proba_v2)

print(f"\n=== Metriken v1 ===")
for k, v in metrics_v1.items():
    print(f"  {k:12s}: {v}")

print(f"\n=== Metriken v2 ===")
for k, v in metrics_v2.items():
    print(f"  {k:12s}: {v}")

✅ Daten geladen:
   X_train: (406, 15)
   X_test:  (102, 15)

✅ Modelle geladen:
   Champion v1: LGBMClassifier
   Champion v2: LGBMClassifier

=== Metriken v1 ===
  precision   : 0.8667
  recall      : 0.8904
  f1          : 0.8784
  roc_auc     : 0.8295
  pr_auc      : 0.9146
  mcc         : 0.5578

=== Metriken v2 ===
  precision   : 0.84
  recall      : 0.863
  f1          : 0.8514
  roc_auc     : 0.8229
  pr_auc      : 0.9077
  mcc         : 0.4593


In [4]:
# =============================================================================
# Zelle 3 – MLflow: Champion v1 loggen
# =============================================================================

# --- Model Signature ---
signature_v1 = infer_signature(X_train, champion_v1.predict(X_train))

with mlflow.start_run(run_name="LGBM_Full_v1_Champion"):

    # --- Tags ---
    mlflow.set_tags({
        "model_type":    "LightGBM",
        "feature_set":   "Full (15 Features)",
        "tuning_method": "Optuna",
        "status":        "Champion",
        "version":       "v1",
        "overfitting":   "Moderat (Gap PR-AUC: 0.131)",
    })

    # --- Parameter loggen ---
    params_v1 = champion_v1.get_params()
    mlflow.log_params(params_v1)

    # --- Metriken loggen ---
    mlflow.log_metrics(metrics_v1)

    # --- Zusätzliche Metriken ---
    mlflow.log_metrics({
        "nested_cv_pr_auc":    0.8693,
        "nested_cv_precision": 0.8295,
        "overfitting_gap_pr_auc":    0.1307,
        "overfitting_gap_precision": 0.1705,
        "bootstrap_ci_precision_low":  0.7808,
        "bootstrap_ci_precision_high": 0.9390,
        "bootstrap_ci_pr_auc_low":     0.8453,
        "bootstrap_ci_pr_auc_high":    0.9672,
        "optimal_threshold_mcc":  0.41,
        "optimal_threshold_bank": 0.90,
    })

    # --- Modell loggen ---
    mlflow.lightgbm.log_model(
        champion_v1,
        artifact_path="model",
        signature=signature_v1,
        registered_model_name="LoanStatus_Champion"
    )

    # --- Artefakte loggen ---
    # Plots
    for plot_file in Path("../reports/figures").glob("*.png"):
        mlflow.log_artifact(str(plot_file), artifact_path="plots")

    # CSVs
    for csv_file in Path("../reports/modeling_csv").glob("*.csv"):
        mlflow.log_artifact(str(csv_file), artifact_path="metrics_csv")

    # Summaries
    for md_file in Path("../reports").glob("*.md"):
        mlflow.log_artifact(str(md_file), artifact_path="summaries")

    # Run ID speichern
    run_id_v1 = mlflow.active_run().info.run_id
    print(f"✅ Champion v1 geloggt.")
    print(f"   Run ID: {run_id_v1}")
    print(f"   Metriken: {metrics_v1}")

2026/04/21 21:34:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 21:34:26 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Champion v1 geloggt.
   Run ID: 0e355454ef0e43eda21fffe906a00fcb
   Metriken: {'precision': 0.8667, 'recall': 0.8904, 'f1': 0.8784, 'roc_auc': 0.8295, 'pr_auc': 0.9146, 'mcc': 0.5578}


Successfully registered model 'LoanStatus_Champion'.
Created version '1' of model 'LoanStatus_Champion'.


In [5]:
# =============================================================================
# Zelle 4 – MLflow: Champion v2 loggen
# =============================================================================

signature_v2 = infer_signature(X_train, champion_v2.predict(X_train))

with mlflow.start_run(run_name="LGBM_Full_v2_GapPenalty"):

    # --- Tags ---
    mlflow.set_tags({
        "model_type":    "LightGBM",
        "feature_set":   "Full (15 Features)",
        "tuning_method": "Optuna + Gap-Penalty",
        "status":        "Verbesserungsversuch",
        "version":       "v2",
        "overfitting":   "Reduziert (Gap Precision: 0.137)",
    })

    # --- Parameter loggen ---
    params_v2 = champion_v2.get_params()
    mlflow.log_params(params_v2)

    # --- Metriken loggen ---
    mlflow.log_metrics(metrics_v2)

    # --- Zusätzliche Metriken ---
    mlflow.log_metrics({
        "nested_cv_pr_auc":          0.8553,
        "nested_cv_precision":       0.8151,
        "overfitting_gap_pr_auc":    0.1324,
        "overfitting_gap_precision": 0.1369,
        "gap_penalty_weight":        0.2,
    })

    # --- Modell loggen ---
    mlflow.lightgbm.log_model(
        champion_v2,
        artifact_path="model",
        signature=signature_v2,
        registered_model_name="LoanStatus_GapPenalty"
    )

    run_id_v2 = mlflow.active_run().info.run_id
    print(f"✅ Champion v2 geloggt.")
    print(f"   Run ID: {run_id_v2}")
    print(f"   Metriken: {metrics_v2}")

2026/04/21 21:35:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/21 21:35:38 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Champion v2 geloggt.
   Run ID: 0fae8f9d9f184020915b7f03464c3e71
   Metriken: {'precision': 0.84, 'recall': 0.863, 'f1': 0.8514, 'roc_auc': 0.8229, 'pr_auc': 0.9077, 'mcc': 0.4593}


Successfully registered model 'LoanStatus_GapPenalty'.
Created version '1' of model 'LoanStatus_GapPenalty'.


In [6]:
# =============================================================================
# Zelle 5 – Champion v1 als Production Model registrieren
# Ziel: Finales Champion-Modell in MLflow Model Registry
#       als "Production" markieren
# =============================================================================

from mlflow import MlflowClient

client = MlflowClient()

# --- Champion v1 als Production setzen ---
client.set_registered_model_alias(
    name="LoanStatus_Champion",
    alias="Production",
    version="1"
)

# --- Beschreibung hinzufügen ---
client.update_registered_model(
    name="LoanStatus_Champion",
    description="""
    Champion Modell – LGBM_Full v1
    Projekt: MSIT Mock Interview – Loan Status Klassifikation

    Metriken (Test-Set):
      Precision:  0.8667
      Recall:     0.8904
      F1:         0.8784
      PR-AUC:     0.9146
      MCC:        0.5578

    Nested CV PR-AUC: 0.8693
    Overfitting Gap:  0.131 (dokumentiert)

    Optimaler Threshold:
      Bankperspektive: 0.90
      Balanced (MCC):  0.41
    """
)

# --- Model Version Beschreibung ---
client.update_model_version(
    name="LoanStatus_Champion",
    version="1",
    description="Finaler Champion – Optuna getunt, 15 Features, is_unbalance=True"
)

print("✅ Champion v1 als Production registriert.")
print(f"\n=== Model Registry Übersicht ===")

# --- Alle registrierten Modelle anzeigen ---
for rm in client.search_registered_models():
    print(f"\nModell: {rm.name}")
    for mv in client.search_model_versions(f"name='{rm.name}'"):
        print(f"  Version: {mv.version} | "
              f"Run ID: {mv.run_id[:8]}...")

✅ Champion v1 als Production registriert.

=== Model Registry Übersicht ===

Modell: LoanStatus_Champion
  Version: 1 | Run ID: 0e355454...

Modell: LoanStatus_GapPenalty
  Version: 1 | Run ID: 0fae8f9d...


In [7]:
# =============================================================================
# Zelle 6 – MLflow Experiment Zusammenfassung
# =============================================================================

# --- Alle Runs anzeigen ---
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.pr_auc DESC"]
)

print("=== MLflow Experiment Runs ===\n")
display_cols = [
    "run_id", "tags.mlflow.runName", "tags.status",
    "metrics.precision", "metrics.recall",
    "metrics.f1", "metrics.pr_auc", "metrics.mcc",
    "metrics.nested_cv_pr_auc",
    "metrics.overfitting_gap_pr_auc"
]

# Nur verfügbare Spalten
available = [c for c in display_cols if c in runs.columns]
print(runs[available].to_string(index=False))

# --- CSV speichern ---
runs.to_csv("../reports/modeling_csv/mlflow_runs.csv", index=False)
print("\n✅ CSV gespeichert: reports/modeling_csv/mlflow_runs.csv")

# --- MLflow UI Hinweis ---
print("""
=============================================================
MLflow UI starten:
  Terminal öffnen → Projektordner navigieren → Befehl:

  mlflow ui --backend-store-uri ../mlruns

  Dann Browser öffnen: http://127.0.0.1:5000
=============================================================
""")

=== MLflow Experiment Runs ===

                          run_id     tags.mlflow.runName          tags.status  metrics.precision  metrics.recall  metrics.f1  metrics.pr_auc  metrics.mcc  metrics.nested_cv_pr_auc  metrics.overfitting_gap_pr_auc
0e355454ef0e43eda21fffe906a00fcb   LGBM_Full_v1_Champion             Champion             0.8667          0.8904      0.8784          0.9146       0.5578                    0.8693                          0.1307
0fae8f9d9f184020915b7f03464c3e71 LGBM_Full_v2_GapPenalty Verbesserungsversuch             0.8400          0.8630      0.8514          0.9077       0.4593                    0.8553                          0.1324

✅ CSV gespeichert: reports/modeling_csv/mlflow_runs.csv

MLflow UI starten:
  Terminal öffnen → Projektordner navigieren → Befehl:

  mlflow ui --backend-store-uri ../mlruns

  Dann Browser öffnen: http://127.0.0.1:5000



In [8]:
# =============================================================================
# Zelle 7 – MLflow Summary Markdown
# Speicherort: reports/mlflow_summary.md
# =============================================================================

summary = """# MLflow Experiment Tracking Summary
**Projekt:** MSIT Mock Interview – Loan Status Klassifikation
**Experiment:** MSIT_MockInterview_LoanStatus
**Tracking URI:** ../mlruns
**MLflow Version:** 3.11.1

---

## 1. Registrierte Modelle

| Modell | Version | Status | Run ID |
|--------|---------|--------|--------|
| LoanStatus_Champion | 1 | **Production** | 0e355454 |
| LoanStatus_GapPenalty | 1 | Archiviert | 0fae8f9d |

---

## 2. Experiment Runs – Vergleich

| Run | Precision | Recall | F1 | PR-AUC | MCC | Nested CV PR-AUC | Gap PR-AUC |
|-----|-----------|--------|-----|--------|-----|-----------------|-----------|
| LGBM_Full_v1_Champion | **0.8667** | **0.8904** | **0.8784** | **0.9146** | **0.5578** | **0.8693** | 0.1307 |
| LGBM_Full_v2_GapPenalty | 0.8400 | 0.8630 | 0.8514 | 0.9077 | 0.4593 | 0.8553 | 0.1324 |

---

## 3. Geloggte Artefakte – Champion v1

| Kategorie | Inhalt |
|-----------|--------|
| `plots/` | Alle Visualisierungen (EDA, Modeling, Evaluation) |
| `metrics_csv/` | Alle CSV Auswertungen |
| `summaries/` | EDA, Preprocessing, Modeling, Tuning, Evaluation Summaries |
| `model/` | LightGBM Modell (MLflow Format) |

---

## 4. Geloggte Parameter – Champion v1

| Parameter | Wert |
|-----------|------|
| n_estimators | 680 |
| max_depth | 14 |
| learning_rate | 0.0314 |
| num_leaves | 58 |
| min_child_samples | 6 |
| subsample | 0.830 |
| colsample_bytree | 0.797 |
| reg_alpha | 7.71e-05 |
| reg_lambda | 9.08 |
| is_unbalance | True |

---

## 5. Geloggte Metriken – Champion v1

### Test-Set Metriken
| Metrik | Wert |
|--------|------|
| precision | 0.8667 |
| recall | 0.8904 |
| f1 | 0.8784 |
| pr_auc | 0.9146 |
| roc_auc | 0.8295 |
| mcc | 0.5578 |

### Robustheit & Stabilität
| Metrik | Wert |
|--------|------|
| nested_cv_pr_auc | 0.8693 |
| nested_cv_precision | 0.8295 |
| overfitting_gap_pr_auc | 0.1307 |
| overfitting_gap_precision | 0.1705 |
| bootstrap_ci_precision_low | 0.7808 |
| bootstrap_ci_precision_high | 0.9390 |
| bootstrap_ci_pr_auc_low | 0.8453 |
| bootstrap_ci_pr_auc_high | 0.9672 |

### Operativer Einsatz
| Metrik | Wert |
|--------|------|
| optimal_threshold_mcc | 0.41 |
| optimal_threshold_bank | 0.90 |

---

## 6. MLflow UI – Anleitung

```bash
# Terminal im Projektordner
mlflow ui --backend-store-uri ../mlruns

# Browser öffnen
http://127.0.0.1:5000
```

Im UI verfügbar:
- Experiment Run Vergleich
- Parameter & Metriken parallel
- Artefakte direkt ansehen
- Model Registry mit Production Tag

---

## 7. Reproduzierbarkeit

| Aspekt | Wert |
|--------|------|
| random_state | 42 |
| CV Strategie | StratifiedKFold (5-Fold) |
| Train/Test Split | 80/20 stratified |
| Datensatz | loans_modified.csv (563 Zeilen original) |
| Preprocessing | 02_preprocessing.ipynb |
| Tuning | Optuna TPESampler (seed=42) |

---

*Erstellt: 06_experiment_tracking.ipynb – MSIT Mock Interview | 2026*
"""

with open("../reports/mlflow_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)

print("✅ MLflow Summary gespeichert: reports/mlflow_summary.md")

✅ MLflow Summary gespeichert: reports/mlflow_summary.md
